# 03a · Style-attack sample

First half of **Stage 4a**. This notebook picks the test articles that will be rewritten and exports them for
the rewriting step. It runs on CPU in about a minute.

For each sampled article it prepares two versions:

| Version | What it is |
|---|---|
| **original** | Title + the first ~300 words of the article (cut at a sentence end) |
| **stripped** | The same, with source markers removed: agency tags like "(Reuters)", publisher suffixes like "- The New York Times", `[VIDEO]` tags, image credits, URLs and tweet embeds |

The **stripped** version is what gets rewritten. Comparing *original → stripped → rewrites* lets us separate
two effects: how much a model relies on **source markers**, and how much on **tone**.

A quick TF‑IDF check at the end previews the source-marker effect.

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
SEED = 42
N_PER_CLASS = 300          # 300 real + 300 fake test articles
MIN_WORDS_ELIGIBLE = 150   # skip very short articles
MAX_WORDS = 300            # truncate longer articles to about this many words
CHUNK_SIZE = 25            # articles per export file for the rewriting step
PROJECT_DIRNAME = "NLP_style_attack"
CODE_URL = "https://raw.githubusercontent.com/AfthabEK/NLP/HEAD/src/fakestyle/text.py"

## 1 · Setup

In [ ]:
import os, sys, json, urllib.request
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("FAKESTYLE_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "models", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# Shared text utilities live in the project repo (src/fakestyle/text.py), so every notebook uses the same code.
CODE_DIR = os.environ.get("FAKESTYLE_CODE_DIR") or "/content/fakestyle_src"
if not os.environ.get("FAKESTYLE_CODE_DIR"):
    os.makedirs(os.path.join(CODE_DIR, "fakestyle"), exist_ok=True)
    open(os.path.join(CODE_DIR, "fakestyle", "__init__.py"), "w").close()
    urllib.request.urlretrieve(CODE_URL, os.path.join(CODE_DIR, "fakestyle", "text.py"))
sys.path.insert(0, CODE_DIR)
from fakestyle.text import truncate_words, strip_source_markers
print("Outputs ->", BASE_DIR)

## 2 · Sample test articles

In [ ]:
test = pd.read_parquet(os.path.join(DIRS["data"], "test.parquet"))
eligible = test[test["n_words"] >= MIN_WORDS_ELIGIBLE]
print(f"Test articles: {len(test):,}; with >= {MIN_WORDS_ELIGIBLE} words: {len(eligible):,}")
print(eligible["fake"].map({0: "real", 1: "fake"}).value_counts().to_string(), "\n")

sample = (pd.concat([eligible[eligible["fake"] == lab].sample(N_PER_CLASS, random_state=SEED) for lab in (0, 1)])
            .sort_values("id").reset_index(drop=True))
print(f"Sampled {len(sample)} articles:", sample["fake"].map({0: "real", 1: "fake"}).value_counts().to_dict())

## 3 · Build the *original* and *stripped* versions

In [ ]:
sample["text_orig"] = sample["text"].map(lambda t: truncate_words(t, MAX_WORDS))
sample["title_orig"] = sample["title"]
stripped = [strip_source_markers(ti, te) for ti, te in zip(sample["title_orig"], sample["text_orig"])]
sample["title_stripped"] = [s[0] for s in stripped]
sample["text_stripped"] = [s[1] for s in stripped]
sample["words_orig"] = sample["text_orig"].str.split().str.len()
sample["words_stripped"] = sample["text_stripped"].str.split().str.len()

changed = (sample["title_orig"] != sample["title_stripped"]) | (sample["text_orig"] != sample["text_stripped"])
print("Share of articles changed by stripping:")
print(changed.groupby(sample["fake"].map({0: "real", 1: "fake"})).mean().round(3).to_string())
print("\nWords per article (original / stripped):")
print(sample.groupby(sample["fake"].map({0: "real", 1: "fake"}))[["words_orig", "words_stripped"]].median().to_string())

In [ ]:
# A few before/after examples of stripping
for _, r in sample[changed].groupby("fake").head(3).iterrows():
    print(f"[{r['id']}] {'FAKE' if r['fake'] else 'REAL'}")
    print(f"  title before: {r['title_orig'][:110]}")
    print(f"  title after : {r['title_stripped'][:110]}")
    print(f"  text before : {r['text_orig'][:160]}...")
    print(f"  text after  : {r['text_stripped'][:160]}...\n")

## 4 · Preview: does stripping source markers hurt TF‑IDF?

The baseline from notebook 01 scored on three inputs for the same 600 articles: the full article, the
truncated original, and the stripped version. DistilBERT is scored on all conditions in notebook 03b (GPU).

In [ ]:
import joblib
from sklearn.metrics import accuracy_score

tfidf = joblib.load(os.path.join(DIRS["models"], "tfidf_lr.joblib"))
def tfidf_predict(titles, texts):
    X = tfidf["vectorizer"].transform([f"{a}\n\n{b}" for a, b in zip(titles, texts)])
    return tfidf["clf"].predict(X)

y = sample["fake"].values
preview = {}
for name, ti, te in [("full article", sample["title"], sample["text"]),
                     ("original (truncated)", sample["title_orig"], sample["text_orig"]),
                     ("stripped", sample["title_stripped"], sample["text_stripped"])]:
    pred = tfidf_predict(ti, te)
    preview[name] = {"accuracy": accuracy_score(y, pred),
                     "acc_real": accuracy_score(y[y == 0], pred[y == 0]),
                     "acc_fake": accuracy_score(y[y == 1], pred[y == 1])}
preview_df = pd.DataFrame(preview).T.round(4)
preview_df

## 5 · Save the sample and export it for rewriting

- `data/attack_sample.parquet`: all versions, used by notebook 03b.
- `data/rewrite_input/chunk_XX.jsonl`: the stripped articles in chunks of 25 (`id`, `fake`, `title`, `text`),
  which are what the rewriting step reads.

In [ ]:
keep = ["id", "fake", "n_words", "title_orig", "text_orig", "title_stripped", "text_stripped",
        "words_orig", "words_stripped"]
sample[keep].to_parquet(os.path.join(DIRS["data"], "attack_sample.parquet"))

export_dir = os.path.join(DIRS["data"], "rewrite_input")
os.makedirs(export_dir, exist_ok=True)
for old in os.listdir(export_dir):
    os.remove(os.path.join(export_dir, old))
records = [{"id": int(r.id), "fake": int(r.fake), "title": r.title_stripped, "text": r.text_stripped}
           for r in sample.itertuples()]
n_chunks = 0
for start in range(0, len(records), CHUNK_SIZE):
    with open(os.path.join(export_dir, f"chunk_{start // CHUNK_SIZE:02d}.jsonl"), "w") as f:
        for rec in records[start:start + CHUNK_SIZE]:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    n_chunks += 1

summary = {
    "notebook": "03a_attack_sample",
    "config": {"SEED": SEED, "N_PER_CLASS": N_PER_CLASS, "MIN_WORDS_ELIGIBLE": MIN_WORDS_ELIGIBLE,
               "MAX_WORDS": MAX_WORDS, "CHUNK_SIZE": CHUNK_SIZE},
    "n_articles": len(sample), "n_chunks": n_chunks,
    "stripping_changed_share": changed.groupby(sample["fake"]).mean().round(4).rename({0: "real", 1: "fake"}).to_dict(),
    "median_words": {"orig": float(sample["words_orig"].median()), "stripped": float(sample["words_stripped"].median())},
    "tfidf_preview": preview_df.to_dict(orient="index"),
}
with open(os.path.join(DIRS["results"], "03a_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f"\nExported {len(records)} articles in {n_chunks} chunks to {export_dir}")

**Done.** Tell Claude the notebook has finished. The rewrites are generated next, then committed to the
project repo, where notebook 03b picks them up.